In [1]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

# model_name = "allegro/herbert-large-cased"
model_name = "dkleczek/bert-base-polish-cased-v1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to("xpu")

max_token_length = 512

In [2]:
def get_embedding(text):
    tokens = tokenizer(text, return_tensors="pt", return_offsets_mapping=True, add_special_tokens=True).to("xpu")
    offsets = tokens.pop("offset_mapping")[0].cpu().tolist()

    with torch.no_grad():
        outputs = model(**tokens)
    embeddings = outputs.last_hidden_state.squeeze(0)

    input_ids = tokens["input_ids"][0].cpu().tolist()
    subtokens = tokenizer.convert_ids_to_tokens(input_ids)

    word_embeddings = []
    words = []

    current_word = ""
    current_vecs = []
    last_end = None

    for subtoken, (start, end), vec in zip(subtokens, offsets, embeddings):
        if start == 0 and end == 0:
            continue

        piece = text[start:end]

        if last_end is None or start == last_end:  
            # kontynuacja tego samego słowa
            current_word += piece
            current_vecs.append(vec)
        else:
            # zapisujemy poprzednie słowo
            if current_vecs:
                word_embeddings.append(torch.mean(torch.stack(current_vecs), dim=0))
                words.append(current_word)

            # zaczynamy nowe słowo
            current_word = piece
            current_vecs = [vec]

        last_end = end

    # dodaj ostatnie słowo
    if current_vecs:
        word_embeddings.append(torch.mean(torch.stack(current_vecs), dim=0))
        words.append(current_word)

    return word_embeddings, words

In [3]:
with open("tekst.txt", "r", encoding="utf-8") as f:
    text = f.read()

seen = set()
unique_words = []
for word in text.split():
    word = word.strip('.,!?;"()[]{}<>«»„”\'`')
    if word.lower() not in seen:
        seen.add(word.lower())
        unique_words.append(word)
text = " ".join(unique_words)

# podziel tekst na kawałki <= max_token_length
def chunk_text(text, max_tokens):
    words = text.split()
    chunks = []
    current_chunk = ""

    for word in words:
        test_chunk = current_chunk + " " + word if current_chunk else word
        tokenized = tokenizer(test_chunk, return_tensors="pt")
        if tokenized.input_ids.size(1) <= max_tokens:
            current_chunk = test_chunk
        else:
            if current_chunk:
                chunks.append(current_chunk)
            current_chunk = word

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

chunks = chunk_text(text, max_token_length - 2)  # -2 na [CLS] i [SEP]

In [4]:
# #maciez podobienstwa
# #nie działa dla większych tekstów

# embedding = []
# words = []

# for chunk in chunks:
#     emb, wds = get_embedding(chunk)
#     embedding.extend(emb)
#     words.extend(wds)

# valid_indices = [i for i, w in enumerate(words) if not w.startswith("##") and w not in ["[CLS]", "[SEP]"]]
# filtered_words = [words[i] for i in valid_indices]
# filtered_embeddings = torch.stack([embedding[i] for i in valid_indices])

# similarity_matrix = F.cosine_similarity(
#     filtered_embeddings.unsqueeze(1),  # [n,1,d]
#     filtered_embeddings.unsqueeze(0),  # [1,n,d]
#     dim=-1
# )

# threshold = 0.85
# visited = set()
# groups = []

# for i in range(len(filtered_words)):
#     if i in visited:
#         continue
#     group = [filtered_words[i]]
#     visited.add(i)
#     for j in range(len(filtered_words)):
#         if j not in visited and similarity_matrix[i, j] > threshold and i != j:
#             group.append(filtered_words[j])
#             visited.add(j)
#     if len(group) > 1:
#         groups.append(group)


# for idx, g in enumerate(groups, 1):
#     print(f"Grupa {idx}: {g}")

In [5]:
# kmeans
from sklearn.cluster import KMeans

embedding = []
words = []

for chunk in chunks:
    emb, wds = get_embedding(chunk)
    embedding.extend(emb)
    words.extend(wds)

valid_indices = [i for i, w in enumerate(words) if not w.startswith("##") and w not in ["[CLS]", "[SEP]"]]
filtered_words = [words[i] for i in valid_indices]
filtered_embeddings = torch.stack([embedding[i] for i in valid_indices]).cpu().numpy()

print(len(filtered_words), filtered_embeddings.shape)

k = 300
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
labels = kmeans.fit_predict(filtered_embeddings)

clusters = {}
for word, label in zip(filtered_words, labels):
    clusters.setdefault(label, []).append(word)

for label, group in clusters.items():
    print(f"Grupa {label}: {group}")

1376 (1376, 768)
Grupa 151: ['po', 'na']
Grupa 159: ['rozpoznaniu', 'zapłatę', 'zastępstwa', 'zasądzenia', 'należytego', 'oddalenie', 'oraz', 'zasądzenie', 'wyniknąć', 'należyte', 'Wypłata', 'uwzględnienie', 'Zamieszczanie']
Grupa 136: ['w', 'przeciwko', 'za', '-', 'netto', '01', 'załącznikami', 'Spór', '1', '2003', 'także', '98', 'wartości', '17,-', 'opłaty', 'mówić']
Grupa 147: ['dniu', 'dnia']
Grupa 60: ['20', '15', '12', '19', '16']
Grupa 69: ['maja', 'października']
Grupa 36: ['2015', '2014', 'akt', 'GC', '286/14', 'ACa', '26/15', 'UZASADNIENIE', 'nr', 'grudnia', 'GNc', '463/13', '1349', 'C', '1513/12', 'CK', '526/04']
Grupa 286: ['r']
Grupa 61: ['Gdańsku', 'S.A', '46', 'k.p.c', 'ust', 'o.o', 'm.in', 'b', 'k.c', 'ustawy', 't.j']
Grupa 242: ['rozprawie', 'sprawy', 'z', 'apelacji', 'sprawie', 'spory']
Grupa 58: ['powództwa', 'powództwo']
Grupa 83: ['spółki', 'Spółce']
Grupa 116: ['ograniczoną', 'odpowiedzialnością', 'Akcyjnej', 'rzecz', 'spółka', 'akcyjna', 'przez', 'Spółką', 'M', '

In [ ]:
#faiss
import faiss

embedding = []
words = []

for chunk in chunks:
    emb, wds = get_embedding(chunk)
    embedding.extend(emb)
    words.extend(wds)

valid_indices = [i for i, w in enumerate(words)
                 if not w.startswith("##") and w not in ["[CLS]", "[SEP]"]]
filtered_words = [words[i] for i in valid_indices]
filtered_embeddings = torch.stack([embedding[i] for i in valid_indices])

emb_norm = F.normalize(filtered_embeddings, p=2, dim=1).cpu().numpy().astype("float32")

index = faiss.IndexFlatIP(emb_norm.shape[1])
index.add(emb_norm)

k = 10  # top k najbardziej podobnych słów
S, I = index.search(emb_norm, k)  # S = podobieństwa, I = indeksy

threshold = 0.65
visited = set()
groups = []

for i in range(len(filtered_words)):
    if i in visited:
        continue
    group = {filtered_words[i]}
    visited.add(i)
    for j, sim in zip(I[i][1:], S[i][1:]):  # pomijamy [0], bo to to samo słowo
        if sim > threshold and j not in visited:
            group.add(filtered_words[j])
            visited.add(j)
    if len(group) > 1:
        groups.append(list(group))

for idx, g in enumerate(groups, 1):
    print(f"Grupa {idx}: {g}")


Grupa 1: ['października', 'maja']
Grupa 2: ['2015', '2014']
Grupa 3: ['Spółce', 'spółki']
Grupa 4: ['S', 'J']
Grupa 5: ['pozwanego', 'pozwana', 'pozwanemu', 'pozwanej', 'Pozwany', 'pozwaną']
Grupa 6: ['Sąd', 'Sądu']
Grupa 7: ['Okręgowego', 'Okręgowy']
Grupa 8: ['IX', 'VI']
Grupa 9: ['GC', 'GNc']
Grupa 10: ['286/14', '26/15', '463/13']
Grupa 11: ['II/', 'I/']
Grupa 12: ['apelację', 'Apelacja']
Grupa 13: ['zasądza', 'zasądził']
Grupa 14: ['powoda', 'powodowi']
Grupa 15: ['kwoty', 'kwotę']
Grupa 16: ['50.000', '2.700', '4.700,00', '94.000']
Grupa 17: ['siedemset', 'tysiące']
Grupa 18: ['kosztów', 'kosztami']
Grupa 19: ['procesowego', 'procesową']
Grupa 20: ['apelacyjne', 'apelacyjnym']
Grupa 21: ['Najpierw', 'oryginale', '232', 'uruchomieniu', 'Reasumując']
Grupa 22: ['zasądzenie', 'zasądzenia']
Grupa 23: ['oddalił', 'wywodził']
Grupa 24: ['należyte', 'należytego']
Grupa 25: ['pisemne', 'pierwsze']
Grupa 26: ['S.A', 'k.p.c', 'o.o']
Grupa 27: ['zgodnie', 'według']
Grupa 28: ['Gwarantowi', 